# exp_19 - Tier 2B: JOINT masked-recon + forecast (track_c, Session 3)

Plan: docs/track_3_rl_reuse/session_plan.md §5 Session 3 step 2. Runs on **local AND Kaggle**.

**Idea (plain):** instead of pretrain-then-finetune (Tier 2A), train ONE model FROM SCRATCH that does
both jobs at once - forecast tomorrow's rain AND reconstruct masked physical values - off a single
shared GAT-GRU encoder. `L = MSE(forecast) + lambda * MSE(masked recon)`.

**This is NOT Tier 2A.** 2A = two-stage (pretrain encoder -> finetune, exp_18). 2B = one-stage joint.
The 2A-vs-2B comparison is the STGCL 'pretrain vs joint' question (no masked-recon weather precedent).

**Masking:** during TRAINING the input window's 6 physical channels are 15% masked and BOTH heads read
that masked input (single shared forward). At VAL/TEST the input is fed CLEAN and only the forecast head
is scored, so RMSE is directly comparable to Tier 0b / Tier 2A. Single seed 42.

In [ ]:
%matplotlib inline

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import glob
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

In [ ]:
# --- environment bootstrap: identical notebook runs on BOTH local and Kaggle ---
if os.path.exists('/kaggle/working'):                       # ---- Kaggle ----
    REPO = '/kaggle/working/Dissertation'
    if not os.path.exists(REPO):
        os.system('git clone --depth 1 https://github.com/itsRkg/Dissertation.git ' + REPO)
    os.system('pip install -q torch_geometric')   # Kaggle lacks PyG (provides GATConv); local already has it
    KAGGLE_DATA_SLUG = 'REPLACE_WITH_YOUR_KAGGLE_DATASET_SLUG'   # e.g. 'itsrkg/dissertation-era5'
else:                                                        # ---- local ----
    REPO = 'C:/Users/rishe/Dissertation'
    KAGGLE_DATA_SLUG = None
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('repo on path:', REPO)

In [ ]:
from utils.data_utils.data_helper_utils import (
    load_pivots, get_lat_lon_aligned, build_edge_index_radius, scale_pivots, temporal_split)
from utils.data_utils.dataset_files.gnn_dataset import (
    build_feature_tensor, build_time_features, add_time_features, SpatioTemporalDataset)
from models.gat_gru_joint import GAT_GRU_Joint
from utils.metric_utils.metrics import rmse, mae, bias, nrmse
from utils.metric_utils.extreme_skill import extreme_skill_table
from utils.metric_utils.embedding_diag import embedding_report, last_gat_embeddings
from utils.env_paths import get_paths

In [ ]:
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 15-feat (anchored to Tier 0b); forecast hyperparams = exp_16 (lr 1e-3, batch 32, 30 ep);
# mask_ratio 0.15 + n_recon 6 = exp_18 pretext; lambda_recon 1.0 (decay to 0.3 if forecast underfits, §5).
CFG = dict(epochs=30, lr=1e-3, batch_size=32, mask_ratio=0.15, lambda_recon=1.0,
           hidden_dim=64, heads=4, seq_len=7, horizon=1, n_recon=6, in_channels=15)
P = get_paths(kaggle_data_slug=KAGGLE_DATA_SLUG)
if P['is_kaggle']:
    hits = glob.glob('/kaggle/input/**/rain_pivot.parquet', recursive=True)
    assert hits, 'rain_pivot.parquet not found under /kaggle/input - attach the dataset'
    DATA = os.path.dirname(hits[0]); P['pivot_dir'] = DATA + '/'
    P['coords_csv'] = glob.glob('/kaggle/input/**/wb_station_coords.csv', recursive=True)[0]
    P['elev_csv']   = glob.glob('/kaggle/input/**/wb_station_elevation.csv', recursive=True)[0]
PIVOT_PATH = P['pivot_dir']
RES_DIR  = P['results_dir'] + '/exp_19_gat_gru_joint_recon/seed_' + str(SEED)
SAVE_DIR = P['models_dir']  + '/exp_19_gat_gru_joint_recon/seed_' + str(SEED)
LOG_DIR  = P['logs_dir']    + '/exp_19_gat_gru_joint_recon/seed_' + str(SEED)
for d in (RES_DIR, SAVE_DIR, LOG_DIR): os.makedirs(d, exist_ok=True)
print(('KAGGLE' if P['is_kaggle'] else 'LOCAL'), '| data', PIVOT_PATH)
print('device', device, '| cfg', CFG)

In [ ]:
# ---- data + graph + scaler fix + 15-feat tensor (IDENTICAL recipe to exp_16 tier0b / exp_18) ----
pivots = load_pivots(PIVOT_PATH)
station_df = pd.read_csv(P['coords_csv']).rename(columns={'latitude':'lat','longitude':'lon'})
elev_df = pd.read_csv(P['elev_csv'])
lat, lon = get_lat_lon_aligned(pivots['rain'], station_df)
edge_index = build_edge_index_radius(lat, lon, threshold_km=100).to(device)
N = pivots['rain'].shape[1]
dates = pivots['rain'].index
T = len(dates); train_end_date = dates[int(0.7*T)-1]
print('train_end_date', train_end_date)
scaled_pivots, scalers = scale_pivots(pivots, train_end=train_end_date)
mu, sigma = scalers['rain']
X_dyn, feature_order = build_feature_tensor(scaled_pivots, use_latent=True)   # 6 physical, rain=idx0
X12 = add_time_features(X_dyn, build_time_features(dates))                    # +6 cyclic time
elev_map = dict(zip(elev_df['station_id'].astype(str), elev_df['elevation_m'].astype(float)))
assert not [c for c in pivots['rain'].columns if str(c) not in elev_map], 'elevation missing'
elev = np.array([elev_map[str(c)] for c in pivots['rain'].columns], dtype=float)
def _z(a):
    a = np.asarray(a, float); s = a.std(); return (a - a.mean())/(s if s else 1.0)
statics = np.stack([_z(lat), _z(lon), _z(elev)], axis=-1)
X15 = np.concatenate([X12, np.broadcast_to(statics[None,:,:], (X12.shape[0], N, 3))], axis=-1)
assert np.allclose(X15[...,0], X12[...,0]) and X15.shape[-1] == 15   # rain stays feature 0
print('X15', X15.shape, '| rain scaler mu=%.4f sigma=%.4f' % (mu, sigma), '| feat0 =', feature_order[0])

In [ ]:
Xtr, Xva, Xte = temporal_split(X15, dates)   # same 70/15/15 chronological split as exp_16
train_loader = DataLoader(SpatioTemporalDataset(Xtr, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'], shuffle=True)
val_loader   = DataLoader(SpatioTemporalDataset(Xva, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'])
test_loader  = DataLoader(SpatioTemporalDataset(Xte, seq_len=CFG['seq_len'], horizon=CFG['horizon']), batch_size=CFG['batch_size'])
print('splits', Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
# ---- masking helper (same as exp_18 pretext) + joint train loop (forecast + masked recon) ----
def mask_input(x, ratio, nrec, gen=None):
    phys = x[..., :nrec]
    if gen is None:
        m = (torch.rand(phys.shape, device=x.device) < ratio)
    else:
        m = (torch.rand(phys.shape, generator=gen) < ratio).to(x.device)   # fixed masks for val (unused: val is clean)
    x_in = x.clone()
    x_in[..., :nrec] = torch.where(m, torch.zeros_like(phys), phys)        # 0 = neutral (z-scored)
    return x_in, phys, m

exp_name = 'exp_19_joint_seed' + str(SEED)
model = GAT_GRU_Joint(in_channels=CFG['in_channels'], hidden_dim=CFG['hidden_dim'],
                      heads=CFG['heads'], n_recon=CFG['n_recon']).to(device)
opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
gscaler = GradScaler(enabled=(device.type=='cuda'))
mse = nn.MSELoss()
LAM = CFG['lambda_recon']
best_val = float('inf'); history = []; t0 = time.time()
for epoch in range(1, CFG['epochs']+1):
    model.train(); tt= tf= tr= 0.0; n=0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x_in, phys, m = mask_input(x, CFG['mask_ratio'], CFG['n_recon'])   # mask the SHARED input
        opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=(device.type=='cuda')):
            fc, rec = model(x_in, edge_index)
            loss_f = mse(fc, y)                                            # forecast: next-day rain
            loss_r = ((rec - phys)[m]**2).mean() if m.sum() > 0 else torch.zeros((), device=device)
            loss = loss_f + LAM * loss_r
        gscaler.scale(loss).backward(); gscaler.step(opt); gscaler.update()
        bs = x.size(0); tt += loss.item()*bs; tf += loss_f.item()*bs; tr += float(loss_r)*bs; n += bs
    tt/=max(n,1); tf/=max(n,1); tr/=max(n,1)
    # validation = FORECAST only, on CLEAN input (matches test-time; this is the checkpoint criterion)
    model.eval(); vf=0.0; nv=0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=(device.type=='cuda')):
                fc, _ = model(x, edge_index)
                vl = mse(fc, y)
            vf += vl.item()*x.size(0); nv += x.size(0)
    vf/=max(nv,1)
    history.append(dict(epoch=epoch, train_total=tt, train_forecast=tf, train_recon=tr, val_forecast=vf))
    print('epoch %02d | total %.5f (fore %.5f + %.2f*recon %.5f) | val_fore %.5f | %.0fs' % (
          epoch, tt, tf, LAM, tr, vf, time.time()-t0))
    if np.isnan(tt) or np.isnan(vf):
        print('  !! NaN - stop and lower lr / batch / lambda_recon (session_plan §9)'); break
    if vf < best_val:
        best_val = vf; torch.save(model.state_dict(), SAVE_DIR + '/' + exp_name + '_best.pt')
        print('  new best val_forecast %.5f -> %s_best.pt' % (vf, exp_name))
torch.save(model.state_dict(), SAVE_DIR + '/' + exp_name + '_last.pt')
pd.DataFrame(history).to_csv(RES_DIR + '/joint_loss_curve.csv', index=False)
print('done. best val forecast MSE = %.5f' % best_val)

In [ ]:
# ---- loss-curve plot + reload best checkpoint ----
h = pd.DataFrame(history)
plt.figure(figsize=(6,4))
plt.plot(h['epoch'], h['train_total'], marker='o', label='train total')
plt.plot(h['epoch'], h['train_forecast'], marker='o', label='train forecast')
plt.plot(h['epoch'], h['train_recon'], marker='o', label='train recon')
plt.plot(h['epoch'], h['val_forecast'], marker='s', label='val forecast (clean)')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Tier 2B joint loss'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig(RES_DIR + '/joint_loss_curve.png', dpi=120); plt.show()
model.load_state_dict(torch.load(SAVE_DIR + '/' + exp_name + '_best.pt', map_location=device))

In [ ]:
def real_metrics(yt, yp):
    return {'RMSE': rmse(yt,yp), 'MAE': mae(yt,yp), 'Bias': bias(yt,yp), 'NRMSE': nrmse(yt,yp)}
SEASONS = {'monsoon':[6,7,8,9], 'non_monsoon':[1,2,3,4,5,10,11,12]}

In [ ]:
# ---- evaluate forecast head on CLEAN input (no masking at inference) ----
@torch.no_grad()
def eval_forecast(loader):
    model.eval(); Ps=[]; Ts=[]; tot=0.0; n=0
    for x, y in loader:
        x = x.to(device)
        fc, _ = model(x, edge_index)
        Ps.append(fc.cpu().numpy()); Ts.append(y.numpy())
        tot += mse(fc, y.to(device)).item()*x.size(0); n += x.size(0)
    return np.concatenate(Ps), np.concatenate(Ts), tot/max(n,1)
preds, targets, _ = eval_forecast(test_loader)
train_loss = eval_forecast(train_loader)[2]
val_loss   = eval_forecast(val_loader)[2]
gap = float(val_loss - train_loss)   # forecast MSE gap at best ckpt (comparable to exp_16)
preds_real = preds*sigma + mu; targets_real = targets*sigma + mu

In [ ]:
# ---- real-space metrics + full save layout (IDENTICAL to exp_16) ----
np.savez_compressed(RES_DIR + '/test_predictions_real.npz', obs=targets_real, pred=preds_real)
ov = real_metrics(targets_real.reshape(-1), preds_real.reshape(-1))
pd.DataFrame([ov]).to_csv(RES_DIR + '/overall_metrics.csv', index=False)
dt = dates[-len(Xte):]; months = dt.month.values[CFG['seq_len']:]
months_flat = np.repeat(months[:,None], preds.shape[1], axis=1).reshape(-1)
seas = []
for sname, sm in SEASONS.items():
    m = np.isin(months_flat, sm)
    r = real_metrics(targets_real.reshape(-1)[m], preds_real.reshape(-1)[m]); r['season'] = sname; seas.append(r)
pd.DataFrame(seas).to_csv(RES_DIR + '/seasonal_metrics_real.csv', index=False)
ids = pivots['rain'].columns.tolist()
st = [dict(real_metrics(targets_real[:,i], preds_real[:,i]), station_id=ids[i]) for i in range(preds.shape[1])]
pd.DataFrame(st).to_csv(RES_DIR + '/station_metrics_real.csv', index=False)
ext = extreme_skill_table(targets_real.reshape(-1), preds_real.reshape(-1))
ext.to_csv(RES_DIR + '/extreme_skill_real.csv', index=False)
e645 = ext[np.isclose(ext['threshold_mm'], 64.5)].iloc[0]
xb = next(iter(test_loader)); xb = xb[0] if isinstance(xb, (list, tuple)) else xb
emb = last_gat_embeddings(model, xb, edge_index, device)
ei_np = edge_index.detach().cpu().numpy()
reps = [embedding_report(emb[b].cpu().numpy(), ei_np) for b in range(emb.shape[0])]
edrep = {k: float(np.mean([r[k] for r in reps])) for k in reps[0]}
pd.DataFrame([edrep]).to_csv(RES_DIR + '/embedding_diag.csv', index=False)
print('RMSE %.3f | MAE %.3f | Bias %.3f | NRMSE %.3f | gap %.4f | CSI@64.5 %.3f' % (
      ov['RMSE'], ov['MAE'], ov['Bias'], ov['NRMSE'], gap, float(e645['CSI'])))

In [ ]:
# ---- one-row summary + comparison vs Tier 0b and (if present) the Tier 2A finetune ----
row = dict(config='ssl_joint_recon', tier='2B', in_channels=15, seed=SEED,
           RMSE=ov['RMSE'], MAE=ov['MAE'], Bias=ov['Bias'], NRMSE=ov['NRMSE'],
           train_loss=float(train_loss), val_loss=float(val_loss), train_val_gap=gap,
           lambda_recon=CFG['lambda_recon'], mask_ratio=CFG['mask_ratio'],
           CSI_64p5=float(e645['CSI']), POD_64p5=float(e645['POD']), n_events_64p5=int(e645['n_events']),
           dirichlet_norm=edrep['dirichlet_energy_norm'], MAD_cosine=edrep['MAD_cosine'], eff_rank=edrep['effective_rank'])
pd.DataFrame([row]).to_csv(RES_DIR + '/joint_summary.csv', index=False)
def _rmse_of(path):
    return float(pd.read_csv(path).iloc[0]['RMSE']) if os.path.exists(path) else None
b0 = _rmse_of(P['results_dir'] + '/exp_16_gat_gru_baseline_seeds/tier0b/seed_42/overall_metrics.csv')
a2 = _rmse_of(P['results_dir'] + '/exp_18_gat_gru_mae_pretrain/finetune/seed_42/overall_metrics.csv')
print('Tier 2B joint RMSE %.3f' % ov['RMSE'])
if b0 is not None: print('  vs Tier 0b   %.3f | delta %+.3f' % (b0, ov['RMSE']-b0))
else:              print('  Tier 0b anchor not found - compare manually to 8.777')
if a2 is not None: print('  vs Tier 2A   %.3f | delta %+.3f  (pretrain-vs-joint = the STGCL question)' % (a2, ov['RMSE']-a2))
else:              print('  Tier 2A finetune not run yet - run exp_18_gat_gru_mae_finetune for the pretrain-vs-joint comparison')
print('reroute gate (session_plan §7): multi-seed candidate if it beats its anchor by >= 0.2 RMSE or shrinks the gap by >= 0.3')

## How to read

- **Headline comparison:** Tier 2B joint RMSE vs **Tier 0b (8.777)** and vs **Tier 2A finetune**. The 2A-vs-2B delta answers 'is the masked-recon signal better as pretraining or as a joint auxiliary loss?' - defensible whichever wins.
- `joint_loss_curve.*`: watch whether the recon term fights the forecast (if `train_forecast` plateaus high while `train_recon` drops, the forecast is underfitting -> lower `lambda_recon` to 0.3 per session_plan §5 and re-run).
- Masking is **train-time only**; val/test use clean input + the forecast head, so RMSE is comparable across tiers.
- `extreme_skill_real.csv` = PRIVATE limitation diagnostic. Single seed = exploratory until Session 5.